# Tomahawk Rev 2.3.1 — Self-Heating Root Search

This Kaggle-safe notebook searches the coupled two-temperature model for zero-auxiliary-power thermal equilibria. It evaluates the proposed 22/18 keV point, includes the laser's absorbed continuous average power in the burn balance while retaining the short pulse as the ignition trigger, treats electron-ion equipartition as an internal exchange term, and checks the eigenvalues of each candidate thermal equilibrium.

Important scope: this is a 0-D/profile-weighted thermal screen. A negative thermal eigenvalue is not an MHD-stability proof. Grad-Shafranov equilibrium, beta limits, RF propagation, alpha kinetics, ash dilution, and nonlinear 3-D MHD remain separate validations.


In [1]:
# CELL 1 — Kaggle-safe imports and run settings
import json, math, os, platform, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
os.environ.setdefault("MPLCONFIGDIR", "/tmp/tomahawk-rev231-root-mpl")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from scipy.optimize import least_squares
    SCIPY_AVAILABLE = True
except Exception as exc:
    SCIPY_AVAILABLE = False
    SCIPY_IMPORT_ERROR = repr(exc)

SEED = int(os.environ.get("TOMAHAWK_ROOT_SEED", "20260825"))
np.random.seed(SEED)

if Path("/kaggle/working").exists():
    OUT = Path(os.environ.get("TOMAHAWK_ROOT_OUTPUT", "/kaggle/working/tomahawk_rev231_root_search"))
else:
    OUT = Path(os.environ.get("TOMAHAWK_ROOT_OUTPUT", str(Path.cwd() / "tomahawk_rev231_root_search")))
OUT = OUT.resolve()
OUT.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("SciPy available:", SCIPY_AVAILABLE)
if not SCIPY_AVAILABLE:
    print("SciPy import error:", SCIPY_IMPORT_ERROR)
print("Output:", OUT)


Python: 3.12.13
SciPy available: True
Output: /kaggle/working/tomahawk_rev231_root_search


In [2]:
# CELL 2 — Physical inputs and the dual-role laser treatment
KEV_TO_J = 1.0e3 * 1.602176634e-19
MEV_TO_J = 1.0e6 * 1.602176634e-19
AMU = 1.66053906660e-27
E_CHARGE = 1.602176634e-19
MU_D, MU_T = 2.01410177812, 3.01604928199
E_FUSION = 17.6 * MEV_TO_J
ALPHA_FRAC = 3.5 / 17.6

CAD_VOLUME_M3 = 950.0
B0_T, BPEAK_COIL_T = 14.0, 35.0
N_D0, N_T0 = 1.20e20, 1.20e20
GAMMA_N, GAMMA_T = 0.70, 0.80
TI_EDGE, TE_EDGE = 1.25, 1.25
Z_EFF = 1.0
TAU_E_S = 3.2
ALPHA_DEPOSITION = 0.90

# The laser has two roles: ignition/profile triggering and continuous average heating.
# The nominal 25 MW/module driver is represented as 20 MW absorbed in the plasma,
# matching the previous coupled notebook's absorbed-power input.
LASER_PULSE_ENERGY_MJ = 2.5
LASER_REPETITION_HZ = 10.0
LASER_NOMINAL_AVG_W = LASER_PULSE_ENERGY_MJ * 1.0e6 * LASER_REPETITION_HZ
LASER_ABSORPTION_EFFICIENCY = 0.80
STEADY_LASER_ABSORBED_W = LASER_NOMINAL_AVG_W * LASER_ABSORPTION_EFFICIENCY

# Baseline alpha partition is a model assumption. The scan exposes the consequence
# of hypothetical alpha channeling; values near 1.0 are not hardware-validated.
ALPHA_TO_ION_BASELINE = 0.15
ALPHA_TO_ION_SCAN = np.linspace(0.15, 1.00, 18)

x, w = np.polynomial.legendre.leggauss(96)
RHO_Q = 0.5 * (x + 1.0)
RHO_W = 0.5 * w * (x + 1.0)

print(json.dumps({
    "laser_nominal_average_MW": LASER_NOMINAL_AVG_W / 1.0e6,
    "laser_absorption_efficiency": LASER_ABSORPTION_EFFICIENCY,
    "steady_laser_absorbed_average_MW": STEADY_LASER_ABSORBED_W / 1.0e6,
    "alpha_to_ion_baseline": ALPHA_TO_ION_BASELINE,
    "alpha_to_ion_scan_max": float(ALPHA_TO_ION_SCAN[-1]),
}, indent=2))


{
  "laser_nominal_average_MW": 25.0,
  "laser_absorption_efficiency": 0.8,
  "steady_laser_absorbed_average_MW": 20.0,
  "alpha_to_ion_baseline": 0.15,
  "alpha_to_ion_scan_max": 1.0
}


In [3]:
# CELL 2A — RF resonance candidates for alpha channeling
# Wavelengths below are vacuum wavelengths. The plasma wavelength depends on the
# cold-plasma dispersion relation and the launched wave vector; wavelength alone
# does not determine alpha-channeling efficiency.
C_LIGHT = 299792458.0
M_ALPHA_AMU = 4.00260325413

def cyclotron_frequency_Hz(B_T, charge_number, mass_amu):
    return charge_number * E_CHARGE * B_T / (2.0 * math.pi * mass_amu * AMU)


alpha_fundamental_Hz = cyclotron_frequency_Hz(B0_T, 2.0, M_ALPHA_AMU)
RF_CANDIDATES = pd.DataFrame([
    {"candidate": "alpha fundamental at plasma B0", "harmonic": 1,
     "B_T": B0_T, "frequency_MHz": alpha_fundamental_Hz / 1.0e6,
     "vacuum_wavelength_m": C_LIGHT / alpha_fundamental_Hz,
     "coupling_note": "alpha cyclotron resonance; requires k-vector and orbit matching"},
    {"candidate": "alpha second harmonic at plasma B0", "harmonic": 2,
     "B_T": B0_T, "frequency_MHz": 2.0 * alpha_fundamental_Hz / 1.0e6,
     "vacuum_wavelength_m": C_LIGHT / (2.0 * alpha_fundamental_Hz),
     "coupling_note": "ion-Bernstein/ICRF harmonic candidate"},
    {"candidate": "alpha third harmonic at plasma B0", "harmonic": 3,
     "B_T": B0_T, "frequency_MHz": 3.0 * alpha_fundamental_Hz / 1.0e6,
     "vacuum_wavelength_m": C_LIGHT / (3.0 * alpha_fundamental_Hz),
     "coupling_note": "ion-Bernstein/ICRF harmonic candidate"},
    {"candidate": "alpha fundamental at coil peak (reference only)", "harmonic": 1,
     "B_T": BPEAK_COIL_T, "frequency_MHz": cyclotron_frequency_Hz(BPEAK_COIL_T, 2.0, M_ALPHA_AMU) / 1.0e6,
     "vacuum_wavelength_m": C_LIGHT / cyclotron_frequency_Hz(BPEAK_COIL_T, 2.0, M_ALPHA_AMU),
     "coupling_note": "reference only; plasma resonance should use B0, not coil surface field"},
])

# Approximate lower-hybrid frequency at the core density; this is principally an
# electron/current-drive coupling candidate, not a direct alpha-channeling proof.
N_CORE_E = N_D0 + N_T0
EPS0 = 8.8541878128e-12
M_E = 9.1093837015e-31
OMEGA_CE = E_CHARGE * B0_T / M_E
OMEGA_CI_D = 2.0 * math.pi * cyclotron_frequency_Hz(B0_T, 1.0, MU_D)
OMEGA_PE = math.sqrt(N_CORE_E * E_CHARGE**2 / (EPS0 * M_E))
OMEGA_LH = 1.0 / math.sqrt(1.0 / (OMEGA_CE * OMEGA_CI_D) + 1.0 / (OMEGA_PE**2))
RF_CANDIDATES = pd.concat([RF_CANDIDATES, pd.DataFrame([{
    "candidate": "lower-hybrid estimate at core density", "harmonic": np.nan,
    "B_T": B0_T, "frequency_MHz": OMEGA_LH / (2.0 * math.pi) / 1.0e6,
    "vacuum_wavelength_m": C_LIGHT / (OMEGA_LH / (2.0 * math.pi)),
    "coupling_note": "current-drive/edge-coupling candidate; alpha coupling requires separate wave model",
}])], ignore_index=True)

print(RF_CANDIDATES.to_string(index=False))


                                      candidate  harmonic  B_T  frequency_MHz  vacuum_wavelength_m                                                                      coupling_note
                 alpha fundamental at plasma B0       1.0 14.0     107.422911             2.790768                    alpha cyclotron resonance; requires k-vector and orbit matching
             alpha second harmonic at plasma B0       2.0 14.0     214.845821             1.395384                                              ion-Bernstein/ICRF harmonic candidate
              alpha third harmonic at plasma B0       3.0 14.0     322.268732             0.930256                                              ion-Bernstein/ICRF harmonic candidate
alpha fundamental at coil peak (reference only)       1.0 35.0     268.557276             1.116307             reference only; plasma resonance should use B0, not coil surface field
          lower-hybrid estimate at core density       NaN 14.0    6460.703858             

In [4]:
# CELL 3 — Reactivity, radiation, equipartition, and species-resolved balances
def dt_reactivity(T_keV):
    # Bosch-Hale-like approximation used by the prior screening notebook.
    # The fit is intentionally bounded to 0.2–100 keV; roots outside that range are rejected.
    T = np.clip(np.asarray(T_keV, dtype=float), 0.2, 100.0)
    BG, MRC2 = 34.3827, 1.124656e6
    C1, C2, C3, C4 = 1.17302e-9, 1.51361e-2, 7.51886e-2, 4.60643e-3
    C5, C6, C7 = 1.35000e-2, -1.06750e-4, 1.36600e-5
    num = T * (C2 + T * (C4 + T * C6))
    den = 1.0 + T * (C3 + T * (C5 + T * C7))
    theta = T / (1.0 - num / den)
    xi = (BG**2 / (4.0 * theta))**(1.0 / 3.0)
    return 1.0e-6 * C1 * theta * np.sqrt(xi / (MRC2 * T**3)) * np.exp(-3.0 * xi)


def coulomb_log(ne_cm3, Te_eV):
    value = 24.0 - np.log(np.maximum(np.sqrt(ne_cm3) / np.maximum(Te_eV, 1.0), 1.0))
    return float(np.clip(value, 10.0, 25.0))


def profile_metrics(Ti0_keV, Te0_keV, tau_E_s=TAU_E_S,
                    alpha_to_ion=ALPHA_TO_ION_BASELINE,
                    P_aux_i_W=0.0, P_e_cooling_W=0.0,
                    P_laser_steady_W=STEADY_LASER_ABSORBED_W):
    reaction_i = brem_i = 0.0
    Wi = We = Pei = 0.0
    dWi_dTi = dWe_dTe = 0.0
    exchange_weight = 0.0

    for rho, weight in zip(RHO_Q, RHO_W):
        shape = max(0.0, 1.0 - rho**2)
        density_shape = shape**GAMMA_N
        temp_shape = shape**GAMMA_T
        nD = N_D0 * density_shape
        nT = N_T0 * density_shape
        ni = nD + nT
        ne = ni
        Ti = TI_EDGE + (Ti0_keV - TI_EDGE) * temp_shape
        Te = TE_EDGE + (Te0_keV - TE_EDGE) * temp_shape
        Te_eV = max(Te * 1.0e3, 200.0)
        lnL = coulomb_log(ne / 1.0e6, Te_eV)

        # Profile-local electron-ion exchange coefficient.
        nu_eD = 3.2e-9 * lnL * (nD / 1.0e6) / (MU_D * Te_eV**1.5)
        nu_eT = 3.2e-9 * lnL * (nT / 1.0e6) / (MU_T * Te_eV**1.5)
        nu_ei = nu_eD + nu_eT

        delta_T = Ti - Te
        reaction_i += weight * nD * nT * dt_reactivity(Ti)
        brem_i += weight * Z_EFF * ne * ni * np.sqrt(Te_eV)
        Wi += weight * 1.5 * ni * Ti * KEV_TO_J
        We += weight * 1.5 * ne * Te * KEV_TO_J
        dWi_dTi += weight * 1.5 * ni * temp_shape * KEV_TO_J
        dWe_dTe += weight * 1.5 * ne * temp_shape * KEV_TO_J
        Pei += weight * 1.5 * ne * delta_T * KEV_TO_J * nu_ei
        exchange_weight += weight * 1.5 * ne * abs(delta_T) * KEV_TO_J

    reaction_rate = CAD_VOLUME_M3 * reaction_i
    P_fusion = reaction_rate * E_FUSION
    P_alpha = P_fusion * ALPHA_FRAC
    P_alpha_dep = P_alpha * ALPHA_DEPOSITION
    P_alpha_i = P_alpha_dep * alpha_to_ion
    P_alpha_e = P_alpha_dep * (1.0 - alpha_to_ion)
    P_brem = CAD_VOLUME_M3 * 5.35e-37 * brem_i
    Wi *= CAD_VOLUME_M3
    We *= CAD_VOLUME_M3
    dWi_dTi *= CAD_VOLUME_M3
    dWe_dTe *= CAD_VOLUME_M3
    Pei *= CAD_VOLUME_M3

    P_transport_i = Wi / max(tau_E_s, 1.0e-9)
    P_transport_e = We / max(tau_E_s, 1.0e-9)
    delta_energy = exchange_weight * CAD_VOLUME_M3
    tau_ei = abs(delta_energy / Pei) if abs(Pei) > 1.0 else float("inf")

    # Species balances. P_ei is internal: it leaves ions and enters electrons.
    ion_balance = P_alpha_i + P_aux_i_W - P_transport_i - Pei
    electron_balance = (P_alpha_e + P_laser_steady_W + Pei
                        - P_brem - P_transport_e - P_e_cooling_W)
    total_balance = (P_alpha_dep + P_laser_steady_W + P_aux_i_W
                     - P_brem - P_transport_i - P_transport_e - P_e_cooling_W)

    return {
        "Ti_keV": float(Ti0_keV), "Te_keV": float(Te0_keV),
        "delta_T_keV": float(Ti0_keV - Te0_keV),
        "P_fusion_W": float(P_fusion), "P_alpha_dep_W": float(P_alpha_dep),
        "P_alpha_i_W": float(P_alpha_i), "P_alpha_e_W": float(P_alpha_e),
        "P_brem_W": float(P_brem), "P_transport_i_W": float(P_transport_i),
        "P_transport_e_W": float(P_transport_e), "P_ei_W": float(Pei),
        "W_i_J": float(Wi), "W_e_J": float(We),
        "dWi_dTi_J_per_keV": float(dWi_dTi), "dWe_dTe_J_per_keV": float(dWe_dTe),
        "tau_ei_effective_s": float(tau_ei),
        "nu_ei_effective_s-1": float(1.0 / tau_ei) if np.isfinite(tau_ei) else 0.0,
        "P_aux_i_required_raw_W": float(P_transport_i + Pei - P_alpha_i),
        "P_e_cooling_required_raw_W": float(P_alpha_e + P_laser_steady_W + Pei - P_brem - P_transport_e),
        "P_aux_i_W": float(P_aux_i_W), "P_e_cooling_W": float(P_e_cooling_W),
        "P_laser_steady_W": float(P_laser_steady_W),
        "ion_balance_W": float(ion_balance), "electron_balance_W": float(electron_balance),
        "total_balance_W": float(total_balance),
        "alpha_self_heating_ratio": float(P_alpha_dep / max(P_brem + P_transport_i + P_transport_e, 1.0)),
    }


def compact_metrics(m):
    keys = ["Ti_keV", "Te_keV", "delta_T_keV", "P_fusion_W", "P_alpha_dep_W",
            "P_brem_W", "P_transport_i_W", "P_transport_e_W", "P_ei_W",
            "tau_ei_effective_s", "alpha_self_heating_ratio", "ion_balance_W",
            "electron_balance_W", "total_balance_W"]
    out = {}
    for key in keys:
        value = m[key]
        out[key.replace("_W", "_GW")] = value / 1.0e9 if key.endswith("_W") else value
    return out


In [5]:
# CELL 4 — Evaluate the proposed 22/18 keV equilibrated point
SWEET_SPOT = profile_metrics(22.0, 18.0, TAU_E_S, ALPHA_TO_ION_BASELINE,
                             P_aux_i_W=0.0, P_e_cooling_W=0.0,
                             P_laser_steady_W=STEADY_LASER_ABSORBED_W)
print(json.dumps(compact_metrics(SWEET_SPOT), indent=2))

print("\nInterpretation:")
print("  The 20 MW absorbed laser average is included as continuous electron heating; the ns/ps pulse timing remains the ignition trigger.")
print("  Equipartition is reduced, but the 22/18 point is not a zero-auxiliary equilibrium at the baseline 15% alpha-to-ion partition.")


{
  "Ti_keV": 22.0,
  "Te_keV": 18.0,
  "delta_T_keV": 4.0,
  "P_fusion_GW": 5.342671801012925,
  "P_alpha_dep_GW": 0.9562168280221995,
  "P_brem_GW": 1.4218251654748657,
  "P_transport_i_GW": 0.15471371631387693,
  "P_transport_e_GW": 0.1273164958762924,
  "P_ei_GW": 0.23209192896729147,
  "tau_ei_effective_s": 0.37774301670191196,
  "alpha_self_heating_ratio": 0.5612077413122938,
  "ion_balance_GW": -0.24337312107783848,
  "electron_balance_GW": -0.484265428564997,
  "total_balance_GW": -0.7276385496428356
}

Interpretation:
  The 20 MW absorbed laser average is included as continuous electron heating; the ns/ps pulse timing remains the ignition trigger.
  Equipartition is reduced, but the 22/18 point is not a zero-auxiliary equilibrium at the baseline 15% alpha-to-ion partition.


In [6]:
# CELL 5 — Correct root residuals and bounded multivariate search
# P_ei must appear with opposite signs in the ion and electron equations.
# It cancels from the summed total-energy equation and must not be double-counted as an external loss.
def thermal_rhs_keV_per_s(y_keV, alpha_to_ion, P_laser_steady_W=STEADY_LASER_ABSORBED_W):
    Ti, Te = np.asarray(y_keV, dtype=float)
    m = profile_metrics(Ti, Te, TAU_E_S, alpha_to_ion,
                        P_aux_i_W=0.0, P_e_cooling_W=0.0,
                        P_laser_steady_W=P_laser_steady_W)
    return np.array([
        m["ion_balance_W"] / max(m["dWi_dTi_J_per_keV"], 1.0),
        m["electron_balance_W"] / max(m["dWe_dTe_J_per_keV"], 1.0),
    ])


def root_residual_GW(y_keV, alpha_to_ion):
    # Scaling in GW keeps the optimizer well-conditioned.
    m = profile_metrics(float(y_keV[0]), float(y_keV[1]), TAU_E_S,
                        alpha_to_ion, P_aux_i_W=0.0, P_e_cooling_W=0.0,
                        P_laser_steady_W=STEADY_LASER_ABSORBED_W)
    return np.array([m["ion_balance_W"], m["electron_balance_W"]]) / 1.0e9


ROOT_SEEDS = np.array([
    [5.0, 5.0], [10.0, 10.0], [15.0, 10.0], [22.0, 18.0],
    [30.0, 20.0], [40.0, 25.0], [55.0, 28.0], [70.0, 30.0], [85.0, 35.0]
])


def thermal_jacobian(y_keV, alpha_to_ion, h_keV=0.03):
    y = np.asarray(y_keV, dtype=float)
    J = np.zeros((2, 2), dtype=float)
    for j in range(2):
        step = np.zeros(2)
        step[j] = h_keV
        J[:, j] = (thermal_rhs_keV_per_s(y + step, alpha_to_ion)
                   - thermal_rhs_keV_per_s(y - step, alpha_to_ion)) / (2.0 * h_keV)
    return J


def find_roots(alpha_to_ion, seeds=ROOT_SEEDS):
    if not SCIPY_AVAILABLE:
        raise RuntimeError("SciPy is required for the bounded root search: " + SCIPY_IMPORT_ERROR)

    roots = []
    for seed in seeds:
        sol = least_squares(
            lambda y: root_residual_GW(y, alpha_to_ion),
            x0=seed,
            bounds=([0.2, 0.2], [100.0, 100.0]),
            max_nfev=160,
            xtol=1.0e-9, ftol=1.0e-9, gtol=1.0e-9,
        )
        y = np.asarray(sol.x, dtype=float)
        residual = root_residual_GW(y, alpha_to_ion)
        if np.linalg.norm(residual) > 1.0e-6:
            continue
        if not np.all(np.isfinite(y)) or np.any(y < 0.2) or np.any(y > 100.0):
            continue
        if any(np.linalg.norm(y - old) < 1.0e-3 for old in roots):
            continue
        roots.append(y)

    rows = []
    for y in roots:
        m = profile_metrics(float(y[0]), float(y[1]), TAU_E_S, alpha_to_ion,
                            P_aux_i_W=0.0, P_e_cooling_W=0.0,
                            P_laser_steady_W=STEADY_LASER_ABSORBED_W)
        J = thermal_jacobian(y, alpha_to_ion)
        eig = np.linalg.eigvals(J)
        rows.append({
            "alpha_to_ion_fraction": float(alpha_to_ion),
            "Ti_keV": float(y[0]), "Te_keV": float(y[1]),
            "delta_T_keV": float(y[0] - y[1]),
            "P_fusion_GW": m["P_fusion_W"] / 1.0e9,
            "P_alpha_dep_GW": m["P_alpha_dep_W"] / 1.0e9,
            "P_brem_GW": m["P_brem_W"] / 1.0e9,
            "P_transport_i_GW": m["P_transport_i_W"] / 1.0e9,
            "P_transport_e_GW": m["P_transport_e_W"] / 1.0e9,
            "P_ei_GW": m["P_ei_W"] / 1.0e9,
            "tau_ei_s": m["tau_ei_effective_s"],
            "alpha_self_heating_ratio": m["alpha_self_heating_ratio"],
            "residual_ion_GW": m["ion_balance_W"] / 1.0e9,
            "residual_electron_GW": m["electron_balance_W"] / 1.0e9,
            "eigenvalue_1_s-1": float(np.real(eig[0])),
            "eigenvalue_2_s-1": float(np.real(eig[1])),
            "max_real_eigenvalue_s-1": float(np.max(np.real(eig))),
            "thermally_stable": bool(np.all(np.real(eig) < 0.0)),
        })
    return rows


In [7]:
# CELL 6 — Root scan: baseline partition and hypothetical alpha-channeling fractions
if not SCIPY_AVAILABLE:
    ROOT_RESULTS = pd.DataFrame()
    print("Root search skipped because SciPy is unavailable.")
else:
    all_rows = []
    for fraction in ALPHA_TO_ION_SCAN:
        all_rows.extend(find_roots(float(fraction)))
    ROOT_RESULTS = pd.DataFrame(all_rows)

    if ROOT_RESULTS.empty:
        print("No zero-auxiliary roots were found in the bounded 0.2–100 keV search.")
    else:
        print(ROOT_RESULTS.to_string(index=False))

    baseline_rows = ROOT_RESULTS[np.isclose(ROOT_RESULTS.alpha_to_ion_fraction, ALPHA_TO_ION_BASELINE)] if not ROOT_RESULTS.empty else pd.DataFrame()
    if baseline_rows.empty:
        print("\nBaseline alpha partition (15%) result: NO zero-auxiliary root found.")
    else:
        print("\nBaseline alpha partition roots:")
        print(baseline_rows.to_string(index=False))

    stable_rows = ROOT_RESULTS[ROOT_RESULTS.thermally_stable] if not ROOT_RESULTS.empty else pd.DataFrame()
    if stable_rows.empty:
        print("No thermally stable root was found in the scanned alpha-channeling range.")
    else:
        first = stable_rows.sort_values(["alpha_to_ion_fraction", "Ti_keV"]).iloc[0]
        best = stable_rows.sort_values(["delta_T_keV", "P_fusion_GW"], ascending=False).iloc[0]
        print("\nFirst stable branch on the scanned alpha-channeling grid:")
        print(first.to_string())
        print("\nHighest-delta stable candidate in the scan (mathematical upper bound):")
        print(best.to_string())


 alpha_to_ion_fraction    Ti_keV    Te_keV  delta_T_keV  P_fusion_GW  P_alpha_dep_GW  P_brem_GW  P_transport_i_GW  P_transport_e_GW  P_ei_GW  tau_ei_s  alpha_self_heating_ratio  residual_ion_GW  residual_electron_GW  eigenvalue_1_s-1  eigenvalue_2_s-1  max_real_eigenvalue_s-1  thermally_stable
                  0.65 46.569661 22.617588    23.952073    11.462015        2.051440   1.589498          0.322999          0.158944 1.010437  0.519553                  0.990345    -1.937706e-10         -2.202355e-10          0.160971         -8.297023                 0.160971             False
                  0.65 57.827627 26.275798    31.551829    12.710018        2.274804   1.710696          0.400108          0.184000 1.078515  0.641202                  0.991285    -3.576279e-15          1.460314e-15         -0.141408         -7.255889                -0.141408              True
                  0.70 42.468423 20.528318    21.940105    10.814088        1.935476   1.515934          0.294908  

In [8]:
# CELL 7 — Diagnostic plots and 22/18 comparison
if not ROOT_RESULTS.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    stable_mask = ROOT_RESULTS.thermally_stable.to_numpy()
    ax[0].scatter(ROOT_RESULTS.loc[~stable_mask, "alpha_to_ion_fraction"], ROOT_RESULTS.loc[~stable_mask, "delta_T_keV"],
                  c="tab:red", label="unstable root")
    ax[0].scatter(ROOT_RESULTS.loc[stable_mask, "alpha_to_ion_fraction"], ROOT_RESULTS.loc[stable_mask, "delta_T_keV"],
                  c="tab:green", label="stable thermal root")
    ax[0].set_xlabel("alpha power fraction directed to ions")
    ax[0].set_ylabel("root temperature split Ti - Te (keV)")
    ax[0].set_title("Zero-auxiliary roots")
    ax[0].grid(alpha=0.25); ax[0].legend()

    ax[1].scatter(ROOT_RESULTS["alpha_to_ion_fraction"], ROOT_RESULTS["max_real_eigenvalue_s-1"],
                  c=np.where(stable_mask, "tab:green", "tab:red"))
    ax[1].axhline(0.0, color="black", lw=1)
    ax[1].set_xlabel("alpha power fraction directed to ions")
    ax[1].set_ylabel("maximum real thermal eigenvalue (s^-1)")
    ax[1].set_title("Thermal stability boundary")
    ax[1].grid(alpha=0.25)
    fig.savefig(OUT / "root_and_stability_scan.png", dpi=170)
    plt.close(fig)

comparison = pd.DataFrame([
    {"case": "baseline 30/10", **compact_metrics(profile_metrics(30.0, 10.0, TAU_E_S, ALPHA_TO_ION_BASELINE, P_laser_steady_W=STEADY_LASER_ABSORBED_W))},
    {"case": "proposed 22/18", **compact_metrics(SWEET_SPOT)},
])
comparison.to_csv(OUT / "sweet_spot_comparison.csv", index=False)
print(comparison.to_string(index=False))


          case  Ti_keV  Te_keV  delta_T_keV  P_fusion_GW  P_alpha_dep_GW  P_brem_GW  P_transport_i_GW  P_transport_e_GW  P_ei_GW  tau_ei_effective_s  alpha_self_heating_ratio  ion_balance_GW  electron_balance_GW  total_balance_GW
baseline 30/10    30.0    10.0         20.0     7.973373        1.427053   1.070776          0.209508          0.072522 2.605719            0.168228                  1.054883       -2.601169             2.695415          0.094246
proposed 22/18    22.0    18.0          4.0     5.342672        0.956217   1.421825          0.154714          0.127316 0.232092            0.377743                  0.561208       -0.243373            -0.484265         -0.727639


In [9]:
# CELL 8 — Export results and manifest
def jsonable(value):
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (bool, str, int, float)) or value is None:
        return value
    return str(value)

target_json = {k: jsonable(v) for k, v in SWEET_SPOT.items()}
(OUT / "sweet_spot_22_18_power_balance.json").write_text(json.dumps(target_json, indent=2), encoding="utf-8")
RF_CANDIDATES.to_csv(OUT / "rf_resonance_candidates.csv", index=False)
if not ROOT_RESULTS.empty:
    ROOT_RESULTS.to_csv(OUT / "self_heating_root_scan.csv", index=False)

stable_count = int(ROOT_RESULTS.thermally_stable.sum()) if not ROOT_RESULTS.empty else 0
manifest = {
    "model_revision": "2.3.1-self-heating-root-search",
    "status": "preliminary 0-D/profile-weighted thermal screen",
    "steady_laser_power_W": STEADY_LASER_ABSORBED_W,
    "laser_role": "ignition/profile trigger plus continuous absorbed average heating",
    "rf_resonance_candidates": "rf_resonance_candidates.csv",
    "sweet_spot": {k: jsonable(v) for k, v in compact_metrics(SWEET_SPOT).items()},
    "root_scan_alpha_to_ion_range": [float(ALPHA_TO_ION_SCAN[0]), float(ALPHA_TO_ION_SCAN[-1])],
    "root_count": int(len(ROOT_RESULTS)),
    "stable_root_count": stable_count,
    "limitations": [
        "P_ei is internal exchange and cancels from the summed total-energy equation.",
        "RF wavelength is a resonance candidate only; coupling efficiency requires a wave-vector, orbit, and antenna model.",
        "Baseline alpha partition is an assumption; high alpha-to-ion fractions are not hardware validated.",
        "No density evolution, alpha ash, impurity transport, RF wave model, NBI model, wall model, or MHD equilibrium.",
        "Negative thermal eigenvalues are not a nonlinear 3-D MHD stability proof.",
        "The reactivity fit is bounded to 0.2–100 keV.",
    ],
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

zip_path = OUT / "Tomahawk_Rev231_SelfHeating_RootSearch_Results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and path != zip_path:
            z.write(path, path.relative_to(OUT))

print("\nFINAL SUMMARY")
print(json.dumps({
    "sweet_spot_bremsstrahlung_GW": SWEET_SPOT["P_brem_W"] / 1.0e9,
    "sweet_spot_P_ei_GW": SWEET_SPOT["P_ei_W"] / 1.0e9,
    "sweet_spot_tau_ei_s": SWEET_SPOT["tau_ei_effective_s"],
    "sweet_spot_alpha_self_heating_ratio": SWEET_SPOT["alpha_self_heating_ratio"],
    "root_count": int(len(ROOT_RESULTS)),
    "stable_root_count": stable_count,
    "results_zip": str(zip_path),
}, indent=2))



FINAL SUMMARY
{
  "sweet_spot_bremsstrahlung_GW": 1.4218251654748657,
  "sweet_spot_P_ei_GW": 0.23209192896729147,
  "sweet_spot_tau_ei_s": 0.37774301670191196,
  "sweet_spot_alpha_self_heating_ratio": 0.5612077413122938,
  "root_count": 16,
  "stable_root_count": 8,
  "results_zip": "/kaggle/working/tomahawk_rev231_root_search/Tomahawk_Rev231_SelfHeating_RootSearch_Results.zip"
}
